In [30]:
from ingest_hw import github_data_reader,build_index
from rag_helper_hw import RAGBase
from openai import OpenAI

In [31]:
open_ai_client = OpenAI()

In [32]:
files = github_data_reader()

In [33]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [34]:
print(f"Q1 - How many lesson pages are in the dataset?\n\nANSWER: {len(files)}")

Q1 - How many lesson pages are in the dataset?

ANSWER: 72


In [35]:
index_op = build_index(documents)

In [36]:
index_results = index_op.search(
    "How does the agentic loop keep calling the model until it stops?",
    num_results=5
)
print(f"Q2 - What's the filename of the first result?\n\nANSWER: {index_results[0]['filename']}")

Q2 - What's the filename of the first result?

ANSWER: 01-agentic-rag/lessons/14-agentic-loop.md


In [37]:
assistant = RAGBase(index = index_op,
                    llm_client = open_ai_client)

In [38]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer.output_text)

The agentic loop keeps calling the model inside a `while True` loop.

Each iteration:
1. Sends the current `messages` back to the model.
2. Checks the response for any `function_call` items.
3. If there are function calls, it runs the tool, appends the tool output to `messages`, and continues.
4. If there are no function calls, it breaks out of the loop.

So the stop condition is:

- **no function calls in the model’s response**

In the code, that’s this check:

```python
if has_function_calls == False:
    break
```

So the model keeps being called until it returns a final answer without asking for any more tools.


In [39]:
print(f' Q3 - Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?\n\nANWER: {answer.usage.input_tokens}')

 Q3 - Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

ANWER: 2345


In [40]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [41]:
print(f"How many chunks do you get?\n\nANSWER: {len(chunks)}")

How many chunks do you get?

ANSWER: 295


In [42]:
#test if input tokens got reduced by using chunks instead of full documents
index_op_chunks = build_index(chunks)
assistant = RAGBase(index = index_op_chunks,
                    llm_client = open_ai_client)
answer_post_chunks = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(f"input tokens after using chunks: {answer_post_chunks.usage.input_tokens}")

input tokens after using chunks: 2238


In [44]:
print(f"Q5 - Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?\n\nANSWER: {answer.usage.input_tokens-answer_post_chunks.usage.input_tokens} fewer input tokens i.e about the same")

Q5 - Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

ANSWER: 107 fewer input tokens i.e about the same
